<a href="https://colab.research.google.com/github/ryo19047/claude1/blob/main/nfl_draft_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!pip -q install lightgbm

from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay

RANDOM_STATE = 42

DATA_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "gci 2026summer/GCIcompetition_1/data"
)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

display(train.head())


In [ ]:
# Basic information
print("Shape:", train.shape)
print("\nColumns:")
print(train.columns.tolist())

print("\nData types:")
train.info()


In [ ]:
# Missing values
missing = pd.DataFrame({
    "Missing Count": train.isna().sum(),
    "Missing %": train.isna().mean() * 100
}).sort_values("Missing %", ascending=False)

display(missing)

In [ ]:
# Target distribution
print("\nDrafted distribution:")
print(train["Drafted"].value_counts())
print(train["Drafted"].value_counts(normalize=True) * 100)

In [ ]:
# Summary statistics
display(train.describe())

In [ ]:
# Drafted vs Not Drafted
plt.figure(figsize=(6, 4))
train["Drafted"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Drafted (0 = No, 1 = Yes)")
plt.ylabel("Number of players")
plt.title("Draft Outcome Distribution")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Numerical feature distributions
numeric_features = train.select_dtypes(
    include="number"
).drop(columns=["Id", "Drafted"], errors="ignore")

numeric_features.hist(figsize=(14, 12), bins=25)
plt.suptitle("Numerical Feature Distributions")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = numeric_features.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label="Correlation")
plt.title("Feature Correlations")
plt.tight_layout()
plt.show()

In [ ]:
# Target
y = train["Drafted"].astype(int)

# Original input features
X_original = train.drop(
    columns=["Id", "School", "Drafted"],
    errors="ignore"
)

# Reserve 20% for final evaluation
dev_idx, holdout_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

y_dev = y.iloc[dev_idx]
y_holdout = y.iloc[holdout_idx]

# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Build a preprocessing + modeling pipeline
def build_pipeline(X, estimator):
    numeric_cols = X.select_dtypes(
        include="number"
    ).columns.tolist()

    categorical_cols = X.select_dtypes(
        exclude="number"
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", estimator)
    ])


In [ ]:
baseline_rf = build_pipeline(
    X_original,
    RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
)

baseline_scores = cross_val_score(
    baseline_rf,
    X_original.iloc[dev_idx],
    y_dev,
    cv=cv,
    scoring="roc_auc"
)

print("Baseline AUC per fold:", np.round(baseline_scores, 4))
print("Mean baseline AUC:", round(baseline_scores.mean(), 4))
print("Standard deviation:", round(baseline_scores.std(), 4))


In [ ]:
combine_cols = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle"
]

def add_features(data):
    data = data.copy()

    data["Combine_Tests_Recorded"] = (
        data[combine_cols].notna().sum(axis=1)
    )

    data["Weight_per_40yd_Time"] = (
        data["Weight"] /
        data["Sprint_40yd"].replace(0, np.nan)
    )

    return data

# Create engineered datasets
train_fe = add_features(train)
test_fe = add_features(test)

X_fe = train_fe.drop(
    columns=["Id", "School", "Drafted"],
    errors="ignore"
)

X_test = test_fe.drop(
    columns=["Id", "School", "Drafted"],
    errors="ignore"
)

# Ensure test features are in the same order
X_test = X_test.reindex(columns=X_fe.columns)

X_dev = X_fe.iloc[dev_idx]
X_holdout = X_fe.iloc[holdout_idx]

# Compare with baseline
rf_engineered = build_pipeline(
    X_fe,
    RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
)

fe_scores = cross_val_score(
    rf_engineered,
    X_dev,
    y_dev,
    cv=cv,
    scoring="roc_auc"
)

print("Original features AUC:", round(baseline_scores.mean(), 4))
print("Engineered features AUC:", round(fe_scores.mean(), 4))
print("Difference:", round(
    fe_scores.mean() - baseline_scores.mean(), 4
))


In [ ]:

model_candidates = {
    "Logistic Regression": build_pipeline(
        X_fe,
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    ),

    "Random Forest": rf_engineered,

    "LightGBM": build_pipeline(
        X_fe,
        lgb.LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=15,
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=1
        )
    ),

    "MLP": build_pipeline(
        X_fe,
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=400,
            early_stopping=True,
            random_state=RANDOM_STATE
        )
    )
}

cv_results = {}

for name, model in model_candidates.items():
    scores = cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="roc_auc"
    )

    cv_results[name] = scores.mean()

    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ("lr", model_candidates["Logistic Regression"]),
        ("rf", model_candidates["Random Forest"]),
        ("lgbm", model_candidates["LightGBM"])
    ],
    voting="soft"
)

ensemble_scores = cross_val_score(
    ensemble,
    X_dev,
    y_dev,
    cv=cv,
    scoring="roc_auc"
)

cv_results["Ensemble"] = ensemble_scores.mean()
model_candidates["Ensemble"] = ensemble

print("Ensemble CV AUC:", round(ensemble_scores.mean(), 4))
print("Best individual model AUC:", round(
    max(v for k, v in cv_results.items() if k != "Ensemble"), 4
))


In [ ]:

lgb_pipeline = build_pipeline(
    X_fe,
    lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=1
    )
)

param_grid = {
    "model__max_depth": [3, 5, 7],
    "model__min_child_samples": [10, 20, 50]
}

grid_search = GridSearchCV(
    estimator=lgb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_dev, y_dev)

print("Best parameters:", grid_search.best_params_)
print("Best CV AUC:", round(grid_search.best_score_, 4))

tuning_results = pd.DataFrame(grid_search.cv_results_)[[
    "param_model__max_depth",
    "param_model__min_child_samples",
    "mean_test_score",
    "std_test_score"
]]

display(tuning_results.sort_values(
    "mean_test_score", ascending=False
))


In [ ]:
# Add tuned LightGBM to our candidates
model_candidates["Tuned LightGBM"] = grid_search.best_estimator_
cv_results["Tuned LightGBM"] = grid_search.best_score_

# Rank candidates
comparison = pd.DataFrame({
    "Model": list(cv_results.keys()),
    "CV AUC": list(cv_results.values())
}).sort_values("CV AUC", ascending=False)

display(comparison)

# Select best by CV score
best_name = comparison.iloc[0]["Model"]
best_model = clone(model_candidates[best_name])

print("Selected model:", best_name)

# Train using development set only
best_model.fit(X_dev, y_dev)

# Predict holdout probabilities
holdout_proba = best_model.predict_proba(X_holdout)[:, 1]

holdout_auc = roc_auc_score(y_holdout, holdout_proba)

print("Final holdout ROC-AUC:", round(holdout_auc, 4))

# Plot ROC curve
RocCurveDisplay.from_predictions(y_holdout, holdout_proba)
plt.title(f"Holdout ROC Curve — {best_name}")
plt.show()


In [ ]:
# Rebuild and train selected model on all labeled data
final_model = clone(model_candidates[best_name])

final_model.fit(X_fe, y)

# Predict draft probabilities for test data
test_predictions = final_model.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(test_predictions))
print("First 10 predictions:", test_predictions[:10])

# Load the competition submission template
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert len(submission) == len(test), "Submission length mismatch"

# Verify row order using IDs when possible
if "Id" in submission.columns and "Id" in test.columns:
    assert submission["Id"].reset_index(drop=True).equals(
        test["Id"].reset_index(drop=True)
    ), "Submission and test IDs are not in the same order"

submission["Drafted"] = test_predictions

# Save as a NEW file; don't overwrite the original template
output_path = DATA_DIR / "submission_my_model.csv"
submission.to_csv(output_path, index=False)

print("Saved to:", output_path)
display(submission.head())


In [ ]:
comparison.to_csv(
    DATA_DIR / "model_comparison.csv",
    index=False
)

In [ ]:
# Compare all models
print("MODEL COMPARISON")
display(comparison)

# Show the selected model
print("\nBest model:", best_name)

# Show the final holdout performance
print(f"Holdout ROC-AUC: {holdout_auc:.4f}")

# Compare training and holdout performance
best_cv_auc = comparison.iloc[0]["CV AUC"]

print(f"Best CV ROC-AUC: {best_cv_auc:.4f}")
print(f"Holdout ROC-AUC: {holdout_auc:.4f}")
print(f"Difference: {best_cv_auc - holdout_auc:.4f}")



## Results and Discussion

Six model configurations were evaluated using
5-fold stratified cross-validation.

### Key Findings

- Random Forest achieved the highest mean
  cross-validation ROC-AUC of 0.7376.
- The ensemble achieved 0.7342 ROC-AUC.
- Hyperparameter-tuned LightGBM achieved
  0.7331 ROC-AUC.
- Random Forest achieved a holdout ROC-AUC
  of 0.7435.

### Interpretation

Random Forest performed best among the tested
configurations. The holdout result was close
to the cross-validation result.

Hyperparameter tuning and ensemble learning
did not outperform the Random Forest model
in this experiment.

### Limitations

- The dataset contains missing measurements.
- Results may depend on the available features.
- External validation on a separate dataset
  has not been performed.
